# Conversor de Base de Votos
Este notebook lê um arquivo `.xlsx`, processa os dados e exporta um `.csv` formatado.

In [1]:
import pandas as pd
from validate_docbr import CPF, CNPJ
from pathlib import Path

# Função para validar documento
def validar_documento(documento):
    cpf = CPF()
    cnpj = CNPJ()
    documento_str = str(documento).strip()

    if len(documento_str) <= 11:
        documento_str = documento_str.zfill(11)
        if cpf.validate(documento_str):
            return 'FISICA'
    elif 12 <= len(documento_str) <= 14:
        documento_str = documento_str.zfill(14)
        if cnpj.validate(documento_str):
            return 'JURIDICA'

    if cpf.validate(documento_str.zfill(11)):
        return 'FISICA'
    elif cnpj.validate(documento_str.zfill(14)):
        return 'JURIDICA'
    else:
        return "Documento inválido"


In [2]:
# Caminho do arquivo .xlsx
caminho_arquivo = Path("modelo_input.xlsx")  # Altere aqui se necessário

# Leitura do arquivo
df1 = pd.read_excel(caminho_arquivo)

# Visualizar colunas disponíveis
df1.head()

,coluna_nome,coluna_documento,coluna_serie_1,coluna_serie_2
0,Fernando Abreu,406.396.370-51,1.0,2
1,Aroldo,491.851.590-85,NaN,50


In [3]:
# Nomes fixos esperados no Excel
coluna_nome = 'coluna_nome'
coluna_documento = 'coluna_documento'
coluna_serie_1 = 'coluna_serie_1'
coluna_serie_2 = 'coluna_serie_2'

# Processar os dados
df2 = pd.DataFrame()
df2['NOME'] = df1[coluna_nome]
df2['TIPO_PESSOA'] = df1[coluna_documento].apply(validar_documento)
df2['CPF_CNPJ'] = df1[coluna_documento]
df2['SERIE_1'] = df1[coluna_serie_1].fillna(0).astype(int)
df2['SERIE_2'] = df1[coluna_serie_2].fillna(0).astype(int)

df2.head()

,NOME,TIPO_PESSOA,CPF_CNPJ,SERIE_1,SERIE_2
0,Fernando Abreu,FISICA,406.396.370-51,1,2
1,Aroldo,FISICA,491.851.590-85,0,50


In [4]:
# Exportar como CSV com nome fixo na mesma pasta
caminho_saida = caminho_arquivo.with_name("base_investidores.csv")
df2.to_csv(caminho_saida, index=False)
print(f"Arquivo exportado para: {caminho_saida}")

Arquivo exportado para: base_investidores.csv
